# 01. Chargement, audit et exploration

Objectif : charger les annotations, auditer leur qualité et explorer la cohorte.

> ℹ️ **Note d'architecture — Pré-traitement du signal :**
> Le pré-traitement (src.preprocess : filtre Butterworth passe-bande 0.5–40 Hz en phase nulle + coupe-bande notch 50 Hz) est automatiquement appliqué lors du chargement des signaux via src.io.load_patient_signal(). Cette encapsulation garantit une stricte cohérence entre l'exploration interactive et les scripts CLI de production (scripts/).

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # racine du projet
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config, io, viz, models, features
from src import postprocess as pp, evaluation as ev
pd.set_option("display.width", 200)
print("Dossier données :", config.DATA_DIR)

Dossier données : C:\Users\MLAB\CRISES ABSENCES


## Annotations et audit

In [2]:
seizures, audit = io.load_annotations()
print("Patients annotés :", audit.n_annotation_patients)
print("Crises retenues  :", audit.n_seizures_retained)
print("Doublons patients:", audit.duplicated_patients)
print("Incohérences N°CE:", audit.nombrece_incoherences)
print("Crises suspectes :", audit.suspect_seizures)
seizures.head()

Patients annotés : 19
Crises retenues  : 82
Doublons patients: ['230406B-A', '230515B-G']
Incohérences N°CE: ['210914B-A : 6 crise(s) annoncée(s), 11 lue(s)']
Crises suspectes :      Patient  Num_Crise  Debut_Sec  Fin_Sec  Duree_Sec
0  210504B-C          1        0.0  21608.0    21608.0


,Patient,Num_Crise,Debut_Sec,Fin_Sec,Duree_Sec
0,190304A-E,1,1291.0,1298.0,7.0
1,191113A-D,1,1254.0,1267.0,13.0
2,191113A-D,2,1428.0,1433.0,5.0
3,210406A-F,1,633.0,641.0,8.0
4,210427B-C,1,1031.0,1049.0,18.0


## Registre patient × fichier

In [3]:
registry = io.build_patient_registry(seizures)
registry

,Patient,Fichier,N_Crises,Duree_Totale_S,Mat_Ok,Ann_Ok
0,190304A-E,190304A_E_0000d.mat,1,7.0,True,True
1,191113A-D,191113A_D_0000d.mat,2,18.0,True,True
2,200625A-F,200625A_F_0000d.mat,0,0.0,True,False
3,210204B-C,210204B_C_0000d.mat,0,0.0,True,False
4,210208B-G,210208B_G_0000d.mat,0,0.0,True,False
5,210406A-F,210406A_F_0000d.mat,1,8.0,True,True
6,210427B-C,210427B_C_0000d.mat,1,18.0,True,True
7,210504B-C,210504B_C_0000d.mat,11,127.0,True,True
8,210914B-A,210914B_A_0000d.mat,11,176.0,True,True
9,210928B-B,210928B_B_0000d.mat,2,55.0,True,True


## Durées des crises

In [4]:
print(seizures.Duree_Sec.describe())
print("CV =", round(seizures.Duree_Sec.std()/seizures.Duree_Sec.mean(), 3))
fig = viz.plot_duration_violin(seizures); viz.save_fig(fig, "01_duration_violin"); plt.show()

count    82.000000
mean     13.317073
std      10.443118
min       2.000000
25%       7.000000
50%      11.000000
75%      16.000000
max      60.000000
Name: Duree_Sec, dtype: float64
CV = 0.784


## Frise chronologique de la cohorte

In [6]:
durations = {p.stem: io.load_patient_signal(p).sig.shape[0]/io.load_patient_signal(p).fs
             for p in io.list_mat_files()}
fig = viz.plot_cohort_timeline(durations, seizures); viz.save_fig(fig, "01_cohort_timeline"); plt.show()